In [1]:
import os
import openai

from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv()) 
openai.api_key = os.environ['OPENAI_API_KEY']

In [2]:
from langchain.agents import tool

## Creating a tool

To create a tool, use the `@tool` decorator.

### Why do we specify data types now?

The `@tool` decorator inspects the function's type hints (like `query: str`) to automatically build the tool's input schema - that's what shows up in `weather.args` below. Without a type hint, LangChain wouldn't know what type of value the LLM is supposed to pass in for that argument, so it couldn't generate a proper schema (or validate the input) for it.


In [3]:
@tool
def weather(query: str) -> str:
    """search weather online"""
    return "30 degrees"

In [4]:
weather.name

'weather'

In [5]:
type(weather)

langchain.tools.base.StructuredTool

In [6]:
weather.args

{'query': {'title': 'Query', 'type': 'string'}}

In [7]:
weather.description

'weather(query: str) -> str - search weather online'

Having a description for each argument (not just the function itself) is better practice, and Pydantic gives you the extra benefit of locking in the data type for each argument.

In [8]:
from pydantic import BaseModel, Field

### Note: mismatched Pydantic schema vs function arguments

If you define more fields in your `args_schema` (the Pydantic class) than your actual function accepts, be careful - this can silently confuse the LLM, since the tool's schema (what the LLM sees and is told it can pass in) won't match what the function actually needs.

That said, when I actually tried calling the function with a mismatch, I got a validation error - so in that case it wasn't silent, it failed loudly and it was easy to tell something was wrong. Still worth double-checking that your `args_schema` and function signature stay in sync, since not every kind of mismatch may fail this cleanly.


In [9]:
class search(BaseModel):
    query: str = Field(description = "name of the city")

In [10]:
@tool(args_schema = search)
def weather_search(query: str) -> str:
    """searches for weather online"""
    return "30 degrees"

In [11]:
weather_search.args

{'query': {'title': 'Query',
  'description': 'name of the city',
  'type': 'string'}}

### What does the `requests` library do?

`requests` is a Python library for making HTTP requests - sending a request to a URL (like an API endpoint) and getting back a response. Here, `requests.get(BASE_URL, params=params)` sends a GET request to the Open-Meteo weather API with our latitude/longitude as query parameters, and `response.json()` parses the JSON response we get back into a Python dict so we can pull the temperature data out of it.


In [12]:
import requests
from pydantic import BaseModel, Field
import datetime

# Define the input schema
class OpenMeteoInput(BaseModel):
    latitude: float = Field(..., description="Latitude of the location to fetch weather data for")
    longitude: float = Field(..., description="Longitude of the location to fetch weather data for")

@tool(args_schema=OpenMeteoInput)
def get_current_temp(latitude: float, longitude: float) -> dict:
    """Fetch current temperature for given coordinates."""
    
    BASE_URL = "https://api.open-meteo.com/v1/forecast"
    
    # Parameters for the request
    params = {
        'latitude': latitude,
        'longitude': longitude,
        'hourly': 'temperature_2m',
        'forecast_days': 1,
    }

    # Make the request
    response = requests.get(BASE_URL, params=params)
    
    if response.status_code == 200:
        results = response.json()
    else:
        raise Exception(f"API Request failed with status code: {response.status_code}")

    current_utc_time = datetime.datetime.utcnow()
    time_list = [datetime.datetime.fromisoformat(time_str.replace('Z', '+00:00')) for time_str in results['hourly']['time']]
    temperature_list = results['hourly']['temperature_2m']
    
    closest_time_index = min(range(len(time_list)), key=lambda i: abs(time_list[i] - current_utc_time))
    current_temperature = temperature_list[closest_time_index]
    
    return f'The current temperature is {current_temperature}°C'

When working with an LLM, make sure it always gets a response either way - if a tool call fails, an API call fails, or something else goes wrong, make sure the LLM knows what happened. Otherwise it may hallucinate, or the response may not be accurate.

Since we're using OpenAI's model, we have to make the tool OpenAI-compatible.

In [13]:
from langchain.tools.render import format_tool_to_openai_function

In [14]:
format_tool_to_openai_function(get_current_temp)

{'name': 'get_current_temp',
 'description': 'get_current_temp(latitude: float, longitude: float) -> dict - Fetch current temperature for given coordinates.',
 'parameters': {'title': 'OpenMeteoInput',
  'type': 'object',
  'properties': {'latitude': {'title': 'Latitude',
    'description': 'Latitude of the location to fetch weather data for',
    'type': 'number'},
   'longitude': {'title': 'Longitude',
    'description': 'Longitude of the location to fetch weather data for',
    'type': 'number'}},
  'required': ['latitude', 'longitude']}}

Since we formatted the tool as an OpenAI function, we can't call it normally anymore - use a dict to pass in the inputs.

In [15]:
get_current_temp({"latitude": 17.3, "longitude": 76.8})

'The current temperature is 27.4°C'

In [16]:
import wikipedia
@tool
def search_wikipedia(query: str) -> str:
    """Run Wikipedia search and get page summaries."""
    page_titles = wikipedia.search(query)
    summaries = []
    for page_title in page_titles[: 3]:
        try:
            wiki_page =  wikipedia.page(title=page_title, auto_suggest=False)
            summaries.append(f"Page: {page_title}\nSummary: {wiki_page.summary}")
        except (
            self.wiki_client.exceptions.PageError,
            self.wiki_client.exceptions.DisambiguationError,
        ):
            pass
    if not summaries:
        return "No good Wikipedia Search Result was found"
    return "\n\n".join(summaries)

In [17]:
format_tool_to_openai_function(search_wikipedia)

{'name': 'search_wikipedia',
 'description': 'search_wikipedia(query: str) -> str - Run Wikipedia search and get page summaries.',
 'parameters': {'title': 'search_wikipediaSchemaSchema',
  'type': 'object',
  'properties': {'query': {'title': 'Query', 'type': 'string'}},
  'required': ['query']}}

In [18]:
print(search_wikipedia({"query": "Agentic AI"}))

Page: AI agent
Summary: An AI agent or agentic AI is an artificial intelligence program that can pursue goals, use software or other tools, and take actions with some level of autonomy. Agentic AI contrasts with tool AI, which performs a narrow, specified task such as answering questions (as with chatbots like ChatGPT) or traditional machine learning algorithms.
While there is no universally agreed-upon definition of an AI agent, common attributes of AI agents include goal-directed behavior, use of external tools, the ability to interact with and modify an external environment, and the ability to autonomously perform multi-step tasks. Their control flow is frequently driven by large language models (LLMs). Agent systems may also include memory components, planning logic, tool interfaces, and orchestration software for coordinating agent components. 
A common application of AI agents is task automation: for example, booking travel plans based on a user's prompted request.

Page: Kimi (A

In [19]:
from langchain.chains.openai_functions.openapi import openapi_spec_to_openai_fn
from langchain.utilities.openapi import OpenAPISpec

## OpenAPI Specification

It's a standard format for describing RESTful APIs - it defines the entire API structure, including the available endpoints, the operations available on each endpoint, the input parameters each operation expects, and the responses it returns.

`openapi_spec_to_openai_fn` is used to convert those RESTful APIs into OpenAI functions.


In [20]:
text = """
{
  "openapi": "3.0.0",
  "info": {
    "version": "1.0.0",
    "title": "Swagger Petstore",
    "license": {
      "name": "MIT"
    }
  },
  "servers": [
    {
      "url": "http://petstore.swagger.io/v1"
    }
  ],
  "paths": {
    "/pets": {
      "get": {
        "summary": "List all pets",
        "operationId": "listPets",
        "tags": [
          "pets"
        ],
        "parameters": [
          {
            "name": "limit",
            "in": "query",
            "description": "How many items to return at one time (max 100)",
            "required": false,
            "schema": {
              "type": "integer",
              "maximum": 100,
              "format": "int32"
            }
          }
        ],
        "responses": {
          "200": {
            "description": "A paged array of pets",
            "headers": {
              "x-next": {
                "description": "A link to the next page of responses",
                "schema": {
                  "type": "string"
                }
              }
            },
            "content": {
              "application/json": {
                "schema": {
                  "$ref": "#/components/schemas/Pets"
                }
              }
            }
          },
          "default": {
            "description": "unexpected error",
            "content": {
              "application/json": {
                "schema": {
                  "$ref": "#/components/schemas/Error"
                }
              }
            }
          }
        }
      },
      "post": {
        "summary": "Create a pet",
        "operationId": "createPets",
        "tags": [
          "pets"
        ],
        "responses": {
          "201": {
            "description": "Null response"
          },
          "default": {
            "description": "unexpected error",
            "content": {
              "application/json": {
                "schema": {
                  "$ref": "#/components/schemas/Error"
                }
              }
            }
          }
        }
      }
    },
    "/pets/{petId}": {
      "get": {
        "summary": "Info for a specific pet",
        "operationId": "showPetById",
        "tags": [
          "pets"
        ],
        "parameters": [
          {
            "name": "petId",
            "in": "path",
            "required": true,
            "description": "The id of the pet to retrieve",
            "schema": {
              "type": "string"
            }
          }
        ],
        "responses": {
          "200": {
            "description": "Expected response to a valid request",
            "content": {
              "application/json": {
                "schema": {
                  "$ref": "#/components/schemas/Pet"
                }
              }
            }
          },
          "default": {
            "description": "unexpected error",
            "content": {
              "application/json": {
                "schema": {
                  "$ref": "#/components/schemas/Error"
                }
              }
            }
          }
        }
      }
    }
  },
  "components": {
    "schemas": {
      "Pet": {
        "type": "object",
        "required": [
          "id",
          "name"
        ],
        "properties": {
          "id": {
            "type": "integer",
            "format": "int64"
          },
          "name": {
            "type": "string"
          },
          "tag": {
            "type": "string"
          }
        }
      },
      "Pets": {
        "type": "array",
        "maxItems": 100,
        "items": {
          "$ref": "#/components/schemas/Pet"
        }
      },
      "Error": {
        "type": "object",
        "required": [
          "code",
          "message"
        ],
        "properties": {
          "code": {
            "type": "integer",
            "format": "int32"
          },
          "message": {
            "type": "string"
          }
        }
      }
    }
  }
}
"""

### What is this?

This is an example OpenAPI spec, written in JSON, for a small demo API called the "Swagger Petstore". It's not real code being run - it's just a specification document describing an imaginary REST API for managing pets, so we can practice converting an OpenAPI spec into OpenAI functions.

It defines:
* `GET /pets` - list all pets (optionally limited to a max count via the `limit` parameter)
* `POST /pets` - create a new pet
* `GET /pets/{petId}` - get info about one specific pet, by its ID
* Plus the shape of the data itself (`Pet`, `Pets`, `Error`) under `components.schemas` - what fields a pet object has (`id`, `name`, `tag`), and what an error response looks like

None of these endpoints actually exist anywhere - it's just a JSON description of what they *would* look like, which `OpenAPISpec.from_text()` and `openapi_spec_to_openai_fn` parse in the next cells.


In [22]:
spec = OpenAPISpec.from_text(text) #converting the text to openapispec first

Attempting to load an OpenAPI 3.0.0 spec.  This may result in degraded performance. Convert your OpenAPI spec to 3.1.* spec for better support.


In [23]:
# now converting that openapispec to openai function
pet_openai_function, pet_callables = openapi_spec_to_openai_fn(spec)

In [24]:
pet_openai_function

[{'name': 'listPets',
  'description': 'List all pets',
  'parameters': {'type': 'object',
   'properties': {'params': {'type': 'object',
     'properties': {'limit': {'type': 'integer',
       'maximum': 100.0,
       'schema_format': 'int32',
       'description': 'How many items to return at one time (max 100)'}},
     'required': []}}}},
 {'name': 'createPets',
  'description': 'Create a pet',
  'parameters': {'type': 'object', 'properties': {}}},
 {'name': 'showPetById',
  'description': 'Info for a specific pet',
  'parameters': {'type': 'object',
   'properties': {'path_params': {'type': 'object',
     'properties': {'petId': {'type': 'string',
       'description': 'The id of the pet to retrieve'}},
     'required': ['petId']}}}}]

### What does `openapi_spec_to_openai_fn` return?

It returns two things:

1. **`pet_openai_function`** - a list of OpenAI function-calling schemas, one per API endpoint (`listPets`, `createPets`, `showPetById`), in the exact `{"name": ..., "description": ..., "parameters": {...}}` format the LLM needs in order to know these operations exist and what arguments each one takes
2. **`pet_callables`** - the actual Python functions that make the real HTTP request to each endpoint. Once the LLM decides which operation to call and with what arguments (using the schema from #1), you'd use the matching entry in `pet_callables` to actually execute that API call

We only use `pet_openai_function` below (to let the model choose a function). `pet_callables` is what you'd reach for next if you wanted to actually execute the API request the model picked.


In [25]:
from langchain.chat_models import ChatOpenAI

In [26]:
model = ChatOpenAI(temperature = 0).bind(functions = pet_openai_function)

In [27]:
model.invoke("what are the three pet names")

AIMessage(content='', additional_kwargs={'function_call': {'name': 'listPets', 'arguments': '{"params":{"limit":3}}'}})

In [28]:
model.invoke("provide me the information about pet with id 42")

AIMessage(content='', additional_kwargs={'function_call': {'name': 'showPetById', 'arguments': '{"path_params":{"petId":"42"}}'}})

## Routing

Sometimes the LLM has to decide which function/tool to use for a given input - that decision is made by the model itself. That's what "routing" refers to here.


In [29]:
functions = [
    format_tool_to_openai_function(f) for f in [
        get_current_temp, search_wikipedia
    ]
]

In [36]:
from langchain.prompts import ChatPromptTemplate

In [31]:
prompt = ChatPromptTemplate.from_messages([
    ("system", "you are an helpful assistant."),
    ("user", "{input}")
]
)

In [32]:
model = ChatOpenAI().bind(functions = functions)

In [33]:
chain = prompt | model

In [34]:
chain.invoke({"input": "what is the weather in SF?"})

AIMessage(content='', additional_kwargs={'function_call': {'name': 'get_current_temp', 'arguments': '{"latitude":37.7749,"longitude":-122.4194}'}})

In [35]:
chain.invoke({"input": "what is langchain?"})

AIMessage(content='', additional_kwargs={'function_call': {'name': 'search_wikipedia', 'arguments': '{"query":"langchain"}'}})

Let's format the output properly.

In [39]:
from langchain.agents.output_parsers import OpenAIFunctionsAgentOutputParser

In [40]:
new_chain = prompt | model | OpenAIFunctionsAgentOutputParser()

In [42]:
result = new_chain.invoke({"input": "what is the weather in New Delhi, India"})

In [43]:
type(result)

langchain.schema.agent.AgentActionMessageLog

You can see the output parser returning an `AgentActionMessageLog`, which indicates a tool is being called.

In [44]:
result.tool

'get_current_temp'

In [45]:
result.tool_input

{'latitude': 28.6139, 'longitude': 77.209, 'units': 'metric'}

In [46]:
get_current_temp(result.tool_input)

'The current temperature is 32.0°C'

In [49]:
result = new_chain.invoke({"input": "Hey, how are you doing"})

In [50]:
type(result)

langchain.schema.agent.AgentFinish

Since this is an `AgentFinish`, we can call `.return_values` to see the output. So the pattern is: if the LLM decides to call a tool, we get back an `AgentActionMessageLog`; if the LLM just returns a plain response (no tool needed), we get back an `AgentFinish`, and its actual content can be viewed via `.return_values`.

In [51]:
result.return_values

{'output': "Hello! I'm here and ready to help. How can I assist you today?"}

Now let's actually call the tools automatically.

In [52]:
from langchain.schema.agent import AgentFinish

In [57]:
def route(message):
    if isinstance(message, AgentFinish):
        return message.return_values["output"]
    
    else:
        tools = {
            "get_current_temp": get_current_temp,
            "search_wikipedia": search_wikipedia
        }
        
        return tools[message.tool].run(message.tool_input)

In [58]:
new_chain = prompt | model | OpenAIFunctionsAgentOutputParser() | route

In [59]:
new_chain.invoke({"input": "what is the current weather in Bengaluru, Karnataka"})

'The current temperature is 25.7°C'

In [74]:
result = new_chain.invoke({"input": "give a brief summary about langchain"})

In [75]:
print(result)

Page: LangChain
Summary: LangChain is a software framework that helps facilitate the integration of large language models (LLMs) into applications. As a language model integration framework, LangChain's use-cases largely overlap with those of language models in general, including document analysis and summarization, chatbots, and code analysis.



Page: List of artificial intelligence companies
Summary: Below is a list of notable companies that primarily focus on artificial intelligence (AI). Companies that simply make use of AI but have a different primary focus are not included.

Page: Vector database
Summary: A vector database, vector store or vector search engine is a database that stores and retrieves embeddings of data in vector space. Vector databases typically implement approximate nearest neighbor algorithms so users can search for records semantically similar to a given input, unlike traditional databases which primarily look up records by exact match. Use-cases for vector da